# SGD by hand with weight decay

Train a softmax classifier with raw tensors: autograd computes the gradients, but you write the update yourself, no `nn.Module` and no `torch.optim`.

Write `manual_sgd(X, labels, W, b, lr, weight_decay, steps)`:

- `X`: float64 `(N, D)`; `labels`: int64 `(N,)` with values in `0..C-1`.
- `W`: float64 `(D, C)`, `b`: float64 `(C,)`: initial weights. Logits are `X @ W + b`.
- `lr`, `weight_decay`: floats; `steps`: int $\ge 1$ (full-batch steps).

## Each step
1. Compute the mean cross-entropy loss $L$ of the logits against `labels` and record it.
2. Get $\nabla_W L$ and $\nabla_b L$ with autograd.
3. Update, decaying the weights but **not** the bias:
$$W \leftarrow W - \text{lr}\,(\nabla_W L + \lambda W), \qquad b \leftarrow b - \text{lr}\,\nabla_b L.$$

## Return
`(losses, W, b)`: float64 `(steps,)` of the loss recorded before each update, and the final weights. Don't modify the input tensors.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import torch.nn.functional as F

def manual_sgd(X, labels, W, b, lr, weight_decay, steps):
    """Return (losses (steps,), W (D, C), b (C,))."""
    W = W.clone().requires_grad_()
    b = b.clone().requires_grad_()
    losses = []
    for _ in range(steps):
        loss = F.cross_entropy(X @ W + b, labels)
        loss.backward()
        with torch.no_grad():
            W -= lr * (W.grad + weight_decay * W)
            b -= lr * b.grad
        losses.append(loss.item())
    return torch.tensor(losses), W.detach(), b.detach()


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0, 1], "dtype": "int64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 1.0, 0.0, 1], "e": {"__tup__": [{"__nd__": [0.6931471805599453], "dtype": "float64"}, {"__nd__": [[0.25, -0.25], [-0.25, 0.25]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.0], [-1.0]], "dtype": "float64"}, {"__nd__": [1, 0], "dtype": "int64"}, {"__nd__": [[0.5, -0.5]], "dtype": "float64"}, {"__nd__": [0.1, 0.0], "dtype": "float64"}, 0.5, 0.1, 3], "e": {"__tup__": [{"__nd__": [1.3142445999237593, 0.8096466008891575, 0.535433148357489], "dtype": "float64"}, {"__nd__": [[-0.3716059268825014, 0.37160592688250144]], "dtype": "float64"}, {"__nd__": [0.07291484124496682, 0.027085158755033215], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.39186984192721785, 0.04992040152334534, -0.16519016568439276], [-0.16796939015046086, 0.06261587751175876, -0.2257110813270981], [0.2969227725915538, -0.0488928601357681, 0.5680696849093915], [-0.6570321512472709, -0.36065998913515956, -0.8560830468147574], [-1.6699613153859187, -2.2324993797182415, 0.06011966573083541], [-0.8305089208950167, -0.7551370241324996, -0.5807141142442936], [0.41772361621879156, 0.5678085702934009, 0.6860170259645172], [0.9247965351503759, 2.9825211297644363, 0.2947190698353022]], "dtype": "float64"}, {"__nd__": [2, 3, 2, 3, 1, 2, 0, 2], "dtype": "int64"}, {"__nd__": [[0.27457863435768987, 0.04463454771797032, -0.008028189799892683, 0.0557088849689639], [-0.2035279763150223, -0.0530607152850418, -0.04054187209770992, 0.173549928115579], [-0.09912403474545024, 0.03630207876346192, 0.07241023328626138, 0.10803153500640994]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, 0.1, 0.0, 5], "e": {"__tup__": [{"__nd__": [1.4141061852472547, 1.3789434632261357, 1.3468330425339397, 1.3175279430510036, 1.2907760569491624], "dtype": "float64"}, {"__nd__": [[0.33604255688840895, -0.013815084994213203, 0.026175133018829465, 0.018491272331706183], [-0.15856861854673474, -0.17141469598594247, 0.09363192444844137, 0.1127707545020408], [-0.05083819589776736, 0.047064266077905585, 0.0830102469283486, 0.03838349520219618]], "dtype": "float64"}, {"__nd__": [-0.052452267973217385, -0.061527952166450794, 0.1172677869574138, -0.0032875668177456303], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.42902964007139516, -0.6870694369346098, -2.176754060698025, 1.1475875882530329], [1.1822000085319717, -0.48993174625983293, 0.2311326140733636, -1.3960603618444776], [-0.20926738944329235, 0.727569904568733, -0.4834008076774047, 0.063086917778475], [1.0999758084019655, 0.6004352300822452, -0.6128311949099872, 1.6147384501822435], [1.7746049045987535, 1.0034774109319793, 0.600669083829522, 0.715768362012354], [0.881849698871554, -1.2626722080098831, 1.079539146589146, -0.8404778884413265], [0.20007064978275851, -0.09887990363746195, 0.4747207458982391, 1.7586005225833379], [0.5328708987333687, -0.11857030102266883, 0.6695577234931344, 0.11588640746924374], [-1.6537599883688663, -1.8111469829351374, -0.5333590032294453, 1.9909196988576745], [-0.2809960162428199, 0.6554372397249413, 1.2773596802356935, -1.330326691881603], [1.347824851639273, 0.8509844414473512, 0.534644509390344, 0.9587398146236379], [-2.0711917249528926, 0.16008423462141808, -1.5632547813990356, 0.32948806583472684]], "dtype": "float64"}, {"__nd__": [1, 0, 1, 2, 1, 1, 2, 1, 2, 0, 0, 1], "dtype": "int64"}, {"__nd__": [[0.10277254501831744, -0.09673380921852592, 0.04148681214636695], [-0.01801890844686472, 0.04934611610907582, -0.11917397367420486], [0.047111115361646805, 6.437947992057309e-05, -0.1170460481432893], [0.06986120607901634, 0.13711069930309436, 0.11537029901339638]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0], "dtype": "float64"}, 0.5, 0.01, 10], "e": {"__tup__": [{"__nd__": [1.0609650655219178, 0.9506665365078099, 0.8796288171254844, 0.8278534512653728, 0.787480176939816, 0.754700846591099, 0.7273621346398996, 0.7041118573200246, 0.6840378306807332, 0.6664925902586306], "dtype": "float64"}, {"__nd__": [[0.2165858132255083, -0.0584013234840698, -0.1129824596339104], [0.2793343906465874, -0.04217849540148086, -0.3207078443277699], [0.3024790334602284, -0.37632735339552986, 0.0073937288687849784], [-0.5180625206084791, -0.05548952894248444, 0.8801349856281988]], "dtype": "float64"}, {"__nd__": [-0.20049024449483618, 0.5810061730994196, -0.38051592860458344], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-2.640597739525694, 0.3481339798862803], [0.3652758862698117, -0.1941492203877895], [-0.17512483422525996, 0.42703368559418636], [0.6741764149976799, -0.7262088928595155], [0.21876308757354118, -0.2365856281025197]], "dtype": "float64"}, {"__nd__": [1, 0, 1, 0, 0], "dtype": "int64"}, {"__nd__": [[-0.17655454461073905, -0.0538426727495635], [-0.1777347309061485, 0.09164696631018031]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 0.2, 0.5, 4], "e": {"__tup__": [{"__nd__": [0.6929867493779296, 0.6075431836965108, 0.5517867393297434, 0.5137337685145531], "dtype": "float64"}, {"__nd__": [[0.1164854425672343, -0.26764905687732876], [-0.23126805250181523, 0.1747858701504005]], "dtype": "float64"}, {"__nd__": [0.07101984507117245, -0.07101984507117244], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0, 0, 1], "dtype": "int64"}, {"__nd__": [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0, 2.0, 3.0], "dtype": "float64"}, 0.3, 0.2, 2], "e": {"__tup__": [{"__nd__": [2.074272631111047, 1.8483184890374655], "dtype": "float64"}, {"__nd__": [[0.8835999999999999, 0.8835999999999999, 0.8835999999999999], [0.8835999999999999, 0.8835999999999999, 0.8835999999999999]], "dtype": "float64"}, {"__nd__": [1.3374410548870979, 2.0431083602046947, 2.6194505849082077], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.28466588496250145, 0.999365093720158, -2.0349888660339843, 1.616047077676221, 0.48050436212616526, -1.1469313179566654], [2.0552518866399643, -0.7201176115220664, -1.152078244650252, -0.16290942428084895, 0.4044055309423483, 0.870220683107881], [2.3443776314293268, -0.8636124415193598, -1.8143544652839232, 0.30994239215887803, -1.2837554276914696, -1.348109890814717], [-0.059724148347244814, 1.329378253797354, -0.8610454249766911, -0.7505748459873494, -0.23590684287511884, -0.35633605762347964], [1.1115734787833915, 0.011307286457126222, -1.2698094722279942, 2.3975450678060177, -0.6391917961646786, -1.6703972795947386], [0.5902490881687265, 0.04229800514213144, 0.6256609015356105, -0.08866562164683132, 0.021256160102546065, 0.16806179314519676], [0.9024537087194603, -2.2240420422838136, 0.1298546149390933, 0.005818243857414263, 2.20848014816363, -0.7422324475696893], [1.2505526451925097, 0.9488321862799959, -0.6853435584819704, 0.6074523532403668, 0.8975466181961613, 1.2998846330442948], [0.9278109178843227, -0.05112231583177016, 0.5642942979215435, -0.6137228930089782, -0.8977828116757061, -0.44132941119447616], [-0.7004907532273561, -0.822115715838066, 1.6712119399490095, -1.7758472622410453, 0.388695527842547, 0.4238674256884224], [-1.1520990178087058, 1.3926105831232898, -0.9800619452299586, 0.22447135563129247, -1.6968179480047645, -1.3850085558621437], [-0.8221243784118001, 0.3023059071870501, 1.5731965163269857, -1.2107593674430017, 0.8561541244309085, -0.5751357564654613], [0.7498604142401586, 0.7330708158946356, 0.2145217715919269, 0.3370841594858764, -1.869929411282161, 0.7208170885720845], [1.2562241901601827, -0.10137053944587548, 0.8448943817745146, 0.7964074239404852, 1.255734682053776, -0.7766187767968843], [-0.5169446917195945, 0.8091554563918911, 0.7745466795394977, 1.439250822484793, -0.23355368006232685, 0.029214724417382716], [-0.703923987433274, -0.2076758457498978, 1.4191381738830848, 0.020448455719568222, 0.5573857883079846, -0.04065486799116721], [-0.2927845168330341, -0.0023945851033794164, -0.023526896510417408, -0.985529823429461, -0.7506902925422764, -0.19037868374320185], [0.5955949905817756, -0.0792540473077412, -1.5753500034371772, 2.6851932470301567, 0.3154488127100096, -0.3146480176752408], [0.11687399256688065, 0.0037161115033039678, 0.04229566974711267, 0.5880272309174134, 0.4845496757204925, 0.17338010823937894], [1.1292192357385296, -0.9267389767998567, 2.1007650353287364, 0.2893732664300419, -0.9746856614091441, -1.9438985471081414]], "dtype": "float64"}, {"__nd__": [4, 4, 0, 1, 0, 3, 1, 1, 4, 3, 4, 1, 4, 1, 4, 2, 0, 3, 4, 0], "dtype": "int64"}, {"__nd__": [[0.009019798180641782, 0.04988808511048159, -0.08353324266185982, 0.08717195161731339, 0.14347465054944836], [-0.21297771188997291, 0.09952937261143134, -0.001793450465179273, -0.004738915397745302, -0.023515379135894136], [-0.033790388937372795, 0.026953886276943645, -0.034717642367758504, -0.2465863548321882, 0.06986385422844517], [0.017370302777261603, 0.06805447306245617, -0.1824355215235925, 0.05052261393952611, -0.0686750146781543], [0.08424969512705861, 0.030667691803627162, 0.1506673709908708, 0.11795053000384129, 0.1110042858574655], [-0.21799671507692106, 0.13191183726548222, 0.06965349691175897, -0.00351584136109807, 0.12752216501805932]], "dtype": "float64"}, {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float64"}, 1.0, 0.001, 15], "e": {"__tup__": [{"__nd__": [1.6032864400401308, 1.3050960469269814, 1.158025406835813, 1.0714085884307178, 1.012698498757271, 0.968757230177617, 0.9336308975014826, 0.9043221110108671, 0.8791679754790325, 0.8571584980029698, 0.8376295284320181, 0.8201150679362602, 0.8042714378731646, 0.7898356514541294, 0.7766009619447822], "dtype": "float64"}, {"__nd__": [[0.35026714240281714, 0.44442017198783357, -0.5848991299684895, -0.18112123845571376, 0.17428551695906688], [-0.7496308665628691, 0.7188892377535439, -0.13770650526192652, -0.4934038207970879, 0.5204933098615672], [-0.24914265472191865, -0.05980245801101015, 0.3232244472816978, 0.048397303385863205, -0.2777019539112946], [-0.06010868177773992, -0.5628570644736371, 0.0508879452028517, 0.2421682853451698, 0.2164617765891948], [-0.884916091480268, 1.573438274382022, 0.307737156889817, 0.1997672093031067, -0.7088533666052527], [-0.9523932302188294, -0.13912190518936782, 0.2627317379499116, 0.5204718704060962, 0.4142840922370193]], "dtype": "float64"}, {"__nd__": [-0.39344344949219756, 0.1235929906807828, -0.6729738222512517, 0.18095079882256296, 0.7618734822401036], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(manual_sgd, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")